# Tag PII columns and define a Unity Catalog ABAC policy that masks them for everyone outside the pii_approved group

A compliance team needs to restrict PII column access (email, phone, dob) to a `pii_approved` group across 40 tables without writing 40 separate column-mask functions, and writing a per-table policy is exactly the maintenance burden the team hired against. ABAC policies are the centralized answer: tag the columns, define one policy, let it apply across the metastore. You have one session to prove the pattern on a single test table with two principal contexts, then write a one-paragraph note for compliance on how to roll the pattern to the other 39 tables.

The hands-on work:

- Create an account-level group `pii_approved` and add yourself.
- Create a second principal (service principal with OAuth M2M credentials) that is NOT in `pii_approved`.
- Create a test table `customer_records` with three PII columns (`email`, `phone`, `dob`) and three non-PII columns (`customer_id`, `signup_date`, `lifetime_value_usd`).
- Tag the three PII columns with `pii=true` via `ALTER TABLE ... ALTER COLUMN ... SET TAGS`.
- Define a Unity Catalog ABAC column-mask policy that fires when a column is tagged `pii=true` AND the principal is NOT in `pii_approved`.
- Query the table as a member (yourself, in this notebook) and as a non-member (the SP, via `databricks-sql-connector`); confirm the PII columns are clear text for the member and `***REDACTED***` for the non-member, while the non-PII columns are unchanged in both contexts.

**Time.** Plan for about 70 minutes hands-on. Most of it is the account-level group/SP setup; the ABAC policy itself is two SQL statements. Budget up to 120 minutes for the session window.

**Cost.** $0.10 to $0.50 per session on your cloud account. The Premium trial is $0 inside the 14-day window; the underlying SQL warehouse spend is what shows up on your bill. Tear down the workspace before day 14 of the trial.

This lab maps to Databricks DE Associate Domain 7 (Governance and Security, ~11 percent provisional).

**Premium trial reminder.** Run Labs 10, 11, and 12 consecutively inside the same 14-day Premium trial. Re-using the trial workspace from Lab 10 is the path of least resistance.

**Rename callout.** If your other prep material says "Attribute-Based Access Control is a 2026 preview," it is now GA on Premium and the exam includes ABAC policies as in-scope content. If your prep material says "column-mask functions" for per-table masking, those still work on the current exam; ABAC policies are the metastore-scoped tag-driven evolution. The exam emphasizes ABAC for the scalability case (40 tables) and per-table column-mask functions for the per-table case.

**Local prerequisites.** This lab needs both workspace-level and account-level credentials. You will paste five getpass prompts: workspace URL, workspace PAT, Databricks account ID (from the account console URL), an account-level admin token (PAT generated in the account console), and the OAuth client_id + client_secret for a service principal you will create in Task 1.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks, the Databricks SDK, and the databricks-sql-connector
# (used in Task 4 for the non-member principal SQL connection).

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0 databricks-sql-connector==3.4.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json as _json
import sys
import time

import requests

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "unity-catalog-abac-column-masking"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_pii"
SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
TABLE_FQN = f"{SCHEMA_FQN}.customer_records"

PII_GROUP_NAME = "pii_approved"
SP_DISPLAY_NAME = "labrun-pii-test-principal"
ABAC_POLICY_NAME = "labrun-pii-mask-policy"
PII_COLUMNS = ("email", "phone", "dob")
NON_PII_COLUMNS = ("customer_id", "signup_date", "lifetime_value_usd")

# Filled by the student.
pii_group_id = None
non_member_sp_application_id = None
non_member_sp_id = None  # internal SCIM id (different from application_id)
member_result = None      # dict of {column_name: value} from the in-notebook query
non_member_result = None  # dict of {column_name: value} from the SP-context query

In [ ]:
# NBVAL_SKIP
# Register the session, validate workspace credentials, capture account
# admin token + account id, expose run_sql() and SCIM helpers.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks workspace PAT: ").strip()
account_host = getpass.getpass("Databricks account console host (e.g. https://accounts.cloud.databricks.com): ").strip()
account_id = getpass.getpass("Databricks account ID (UUID from the account console URL): ").strip()
account_token = getpass.getpass("Databricks account admin PAT: ").strip()

for name, val in (
    ("workspace URL", databricks_host),
    ("workspace PAT", databricks_token),
    ("account host", account_host),
    ("account id", account_id),
    ("account PAT", account_token),
):
    if not val:
        print(f"{name} is required. Re-run this cell.")
        raise SystemExit(1)

if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
databricks_host = databricks_host.rstrip("/")
if not account_host.startswith("https://"):
    account_host = f"https://{account_host}"
account_host = account_host.rstrip("/")

w = WorkspaceClient(host=databricks_host, token=databricks_token)
try:
    me = w.current_user.me()
except Exception as exc:
    print("Workspace credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Create one and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
WAREHOUSE_HTTP_PATH = (
    WAREHOUSE.odbc_params.path
    if (WAREHOUSE.odbc_params and WAREHOUSE.odbc_params.path)
    else f"/sql/1.0/warehouses/{WAREHOUSE_ID}"
)
print(f"SQL warehouse: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
    "warehouse_http_path": WAREHOUSE_HTTP_PATH,
    "account_host": account_host,
    "account_id": account_id,
    "account_token": account_token,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180, ignore_errors=False):
    wid = warehouse_id or WAREHOUSE_ID
    try:
        resp = w.statement_execution.execute_statement(
            warehouse_id=wid, statement=statement, wait_timeout="50s",
        )
        statement_id = resp.statement_id
        deadline = time.time() + wait_seconds
        while True:
            state = resp.status.state if resp.status else None
            if state == StatementState.SUCCEEDED:
                break
            if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
                err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
                if ignore_errors:
                    return {"columns": [], "rows": [], "statement_id": statement_id, "error": err}
                raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement[:200]}")
            if time.time() > deadline:
                raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
            time.sleep(2)
            resp = w.statement_execution.get_statement(statement_id)
        columns = []
        if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
            columns = [c.name for c in resp.manifest.schema.columns]
        rows = []
        if resp.result and resp.result.data_array:
            rows = list(resp.result.data_array)
        return {"columns": columns, "rows": rows, "statement_id": statement_id, "error": None}
    except Exception as exc:
        if ignore_errors:
            return {"columns": [], "rows": [], "statement_id": None, "error": str(exc)}
        raise


def scim_get(path, params=None):
    url = f"{account_host}/api/2.0/accounts/{account_id}/scim/v2/{path}"
    headers = {"Authorization": f"Bearer {account_token}", "Accept": "application/scim+json"}
    resp = requests.get(url, headers=headers, params=params or {}, timeout=30)
    return resp


def scim_post(path, body):
    url = f"{account_host}/api/2.0/accounts/{account_id}/scim/v2/{path}"
    headers = {
        "Authorization": f"Bearer {account_token}",
        "Content-Type": "application/scim+json",
    }
    resp = requests.post(url, headers=headers, data=_json.dumps(body), timeout=30)
    return resp


def scim_patch(path, body):
    url = f"{account_host}/api/2.0/accounts/{account_id}/scim/v2/{path}"
    headers = {
        "Authorization": f"Bearer {account_token}",
        "Content-Type": "application/scim+json",
    }
    resp = requests.patch(url, headers=headers, data=_json.dumps(body), timeout=30)
    return resp


def scim_delete(path):
    url = f"{account_host}/api/2.0/accounts/{account_id}/scim/v2/{path}"
    headers = {"Authorization": f"Bearer {account_token}"}
    return requests.delete(url, headers=headers, timeout=30)


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest + adapter + atexit + orphan scan. The ABAC policy
# tears down first because a stale policy keeps applying silently;
# tags follow; the table and schema cascade; the SP and group are
# account-level cleanups via SCIM.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_abac_policy",
        id=ABAC_POLICY_NAME,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP POLICY IF EXISTS {ABAC_POLICY_NAME}\"",
    ),
] + [
    CleanupResource(
        type="uc_column_tag",
        id=f"{TABLE_FQN}.{col}/pii",
        region="databricks",
        cli_delete_command=(
            f"databricks sql -e \"ALTER TABLE {TABLE_FQN} ALTER COLUMN {col} UNSET TAGS ('pii')\""
        ),
    )
    for col in PII_COLUMNS
] + [
    CleanupResource(
        type="uc_managed_table",
        id=TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {SCHEMA_FQN} CASCADE\"",
    ),
    CleanupResource(
        type="service_principal",
        id=SP_DISPLAY_NAME,
        region="databricks",
        cli_delete_command=f"databricks account service-principals delete {SP_DISPLAY_NAME}",
    ),
    CleanupResource(
        type="account_group",
        id=PII_GROUP_NAME,
        region="databricks",
        cli_delete_command=f"databricks account groups delete {PII_GROUP_NAME}",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_abac_policy":
            run_sql(f"DROP POLICY IF EXISTS {rid}", ignore_errors=True)
        elif rtype == "uc_column_tag":
            # rid format: TABLE_FQN.column/tagkey
            tbl, col_tag = rid.rsplit(".", 1)
            col, tagkey = col_tag.split("/", 1)
            run_sql(
                f"ALTER TABLE {tbl} ALTER COLUMN {col} UNSET TAGS ('{tagkey}')",
                ignore_errors=True,
            )
        elif rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}", ignore_errors=True)
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE", ignore_errors=True)
        elif rtype == "service_principal":
            # Look up by display name then DELETE.
            resp = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{rid}"'})
            if resp.status_code == 200:
                for sp in (resp.json().get("Resources") or []):
                    sp_id = sp.get("id")
                    if sp_id:
                        scim_delete(f"ServicePrincipals/{sp_id}")
        elif rtype == "account_group":
            resp = scim_get("Groups", params={"filter": f'displayName eq "{rid}"'})
            if resp.status_code == 200:
                for g in (resp.json().get("Resources") or []):
                    g_id = g.get("id")
                    if g_id:
                        scim_delete(f"Groups/{g_id}")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_abac_policy":
            res = run_sql(
                "SELECT 1 FROM system.information_schema.policies "
                f"WHERE policy_name = '{rid}'",
                ignore_errors=True,
            )
            return len(res.get("rows", []) or []) > 0
        if rtype == "uc_column_tag":
            tbl, col_tag = rid.rsplit(".", 1)
            col, tagkey = col_tag.split("/", 1)
            cat, par, tname = tbl.split(".")
            res = run_sql(
                "SELECT 1 FROM system.information_schema.column_tags "
                f"WHERE catalog_name = '{cat}' AND schema_name = '{par}' "
                f"AND table_name = '{tname}' AND column_name = '{col}' "
                f"AND tag_name = '{tagkey}'",
                ignore_errors=True,
            )
            return len(res.get("rows", []) or []) > 0
        if rtype == "uc_managed_table":
            cat, par, tname = rid.split(".")
            res = run_sql(
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
                f"AND table_name = '{tname}'"
            )
            return len(res["rows"]) > 0
        if rtype == "uc_schema":
            cat, par = rid.split(".")
            res = run_sql(
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{cat}' AND schema_name = '{par}'"
            )
            return len(res["rows"]) > 0
        if rtype == "service_principal":
            resp = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{rid}"'})
            return resp.status_code == 200 and bool((resp.json().get("Resources") or []))
        if rtype == "account_group":
            resp = scim_get("Groups", params={"filter": f'displayName eq "{rid}"'})
            return resp.status_code == 200 and bool((resp.json().get("Resources") or []))
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        # Schema/table-tag scans (same shape as Lab 9 / Lab 10).
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"schema:{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"table:{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql, ignore_errors=True)
            except Exception:
                continue
            for row in (result.get("rows") or []):
                arns.append(fmt(row))
        # SCIM scans for residual labrun- groups and SPs.
        try:
            resp = scim_get("Groups", params={"filter": f'displayName eq "{PII_GROUP_NAME}"'})
            if resp.status_code == 200:
                for g in (resp.json().get("Resources") or []):
                    arns.append(f"group:{g.get('displayName')}")
        except Exception:
            pass
        try:
            resp = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'})
            if resp.status_code == 200:
                for sp in (resp.json().get("Resources") or []):
                    arns.append(f"sp:{sp.get('displayName')}")
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the pii_approved group, add yourself, and create a non-member service principal

Three SCIM operations against the account API. The notebook ships helpers (`scim_get`, `scim_post`, `scim_patch`, `scim_delete`) so you do not have to compose URLs by hand.

**Step A.** Create the account-level group `pii_approved`.

```python
group_resp = scim_post("Groups", {
    "schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"],
    "displayName": PII_GROUP_NAME,
})
```

If the group already exists from a previous attempt, scim_post returns 409; the lab tolerates that by re-fetching the existing group.

**Step B.** Add yourself to the group.

Look up your account-level user id (the SCIM Users endpoint, filtered by `userName eq "<your email>"`), then PATCH the group to add the member.

**Step C.** Create the non-member service principal `labrun-pii-test-principal` and generate an OAuth M2M secret for it (do this in the account console; the SCIM API creates the SP but the OAuth secret CREATE is via the account-side `accounts/{id}/servicePrincipals/{application_id}/credentials/secrets` endpoint).

You will paste the SP's `client_id` (which is the `applicationId` SCIM returns) and `client_secret` later in Task 4. For Checkpoint 1, only the SP's existence and group-non-membership matter.

In [ ]:
# NBVAL_SKIP
# Task 1: Create pii_approved group, add the student, create non-member SP.

global pii_group_id, non_member_sp_application_id, non_member_sp_id

# YOUR CODE: Create the group via scim_post("Groups", {...}). On 409, fetch
# the existing group via scim_get("Groups", params={"filter": ...}) and
# extract its id.
# Example:
#   resp = scim_post("Groups", {
#       "schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"],
#       "displayName": PII_GROUP_NAME,
#   })
#   if resp.status_code == 201:
#       pii_group_id = resp.json()["id"]
#   else:
#       lookup = scim_get("Groups", params={"filter": f'displayName eq "{PII_GROUP_NAME}"'})
#       pii_group_id = lookup.json()["Resources"][0]["id"]

if not pii_group_id:
    print("pii_group_id is None. Run the group create above.")
    raise SystemExit(1)

# YOUR CODE: Look up your user id by your email/userName and PATCH the
# group to add the member.
# Example:
#   user_lookup = scim_get("Users", params={"filter": f'userName eq "{CALLER_USER}"'})
#   my_user_id = user_lookup.json()["Resources"][0]["id"]
#   scim_patch(
#       f"Groups/{pii_group_id}",
#       {
#           "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
#           "Operations": [{"op": "add", "path": "members", "value": [{"value": my_user_id}]}],
#       },
#   )

# YOUR CODE: Create the non-member service principal via scim_post.
# Example:
#   sp_resp = scim_post("ServicePrincipals", {
#       "schemas": ["urn:ietf:params:scim:schemas:core:2.0:ServicePrincipal"],
#       "displayName": SP_DISPLAY_NAME,
#       "active": True,
#   })
#   if sp_resp.status_code == 201:
#       sp_payload = sp_resp.json()
#       non_member_sp_id = sp_payload["id"]
#       non_member_sp_application_id = sp_payload.get("applicationId")
#   else:
#       lookup = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'})
#       sp_payload = lookup.json()["Resources"][0]
#       non_member_sp_id = sp_payload["id"]
#       non_member_sp_application_id = sp_payload.get("applicationId")

if not non_member_sp_application_id:
    print("non_member_sp_application_id is None. Create the SP and capture applicationId.")
    raise SystemExit(1)

print(f"pii_approved group id:           {pii_group_id}")
print(f"Non-member SP id (SCIM internal): {non_member_sp_id}")
print(f"Non-member SP applicationId:     {non_member_sp_application_id}")
print()
print("Generate an OAuth M2M client secret for this SP in the account console:")
print(f"  {account_host}/account/serviceprincipals/{non_member_sp_application_id}/credentials")
print("Copy the client_id (= applicationId above) and the client_secret;")
print("you will paste both in Task 4.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: pii_approved group exists with the student as member.
# Non-member SP exists in the account directory but is NOT in the group.


def checkpoint_1(session):
    try:
        if not pii_group_id:
            return CheckpointResult(
                status="fail",
                error_reason="pii_group_id is None. Create the group via scim_post.",
            )

        group_resp = scim_get(f"Groups/{pii_group_id}")
        if group_resp.status_code != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GET /Groups/{pii_group_id} returned {group_resp.status_code}. "
                    f"The group may not exist or the account token may not have scope."
                ),
            )
        group = group_resp.json()
        if group.get("displayName") != PII_GROUP_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Group displayName is {group.get('displayName')!r}; expected "
                    f"{PII_GROUP_NAME!r}."
                ),
            )

        members = group.get("members") or []
        member_ids = {m.get("value") for m in members}
        # Look up the student's user id and confirm membership.
        user_resp = scim_get(
            "Users",
            params={"filter": f'userName eq "{CALLER_USER}"'},
        )
        if user_resp.status_code != 200 or not (user_resp.json().get("Resources") or []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not look up your account user (userName={CALLER_USER}). "
                    f"Confirm the account token has account admin scope."
                ),
            )
        my_user_id = user_resp.json()["Resources"][0]["id"]
        if my_user_id not in member_ids:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Your account user id {my_user_id!r} is not in {PII_GROUP_NAME!r} "
                    f"members. PATCH the group to add yourself."
                ),
            )

        if not non_member_sp_application_id:
            return CheckpointResult(
                status="fail",
                error_reason="non_member_sp_application_id is None. Create the SP.",
            )
        sp_resp = scim_get(
            "ServicePrincipals",
            params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'},
        )
        if sp_resp.status_code != 200 or not (sp_resp.json().get("Resources") or []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not find SP with displayName={SP_DISPLAY_NAME!r}. "
                    f"scim_post('ServicePrincipals', ...) the principal first."
                ),
            )
        sp = sp_resp.json()["Resources"][0]
        sp_internal_id = sp.get("id")
        if sp_internal_id in member_ids:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"The SP is in {PII_GROUP_NAME!r}; the lab needs it OUTSIDE the "
                    f"group so the masking comparison is meaningful. Remove it."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: pii_group_id, the membership PATCH, the SP, or accidental SP membership in the group. Each is one SCIM call.

</details>

<details><summary>Hint 2 (direction)</summary>

Three account-API operations: create the group via `scim_post("Groups", ...)`, fetch your user id via `scim_get("Users", params={"filter": f'userName eq "{CALLER_USER}"'})`, PATCH the group to add a `members` value of your user id, then create the SP via `scim_post("ServicePrincipals", ...)`. Capture the SP's `applicationId` (this is the OAuth `client_id` you'll use in Task 4).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global pii_group_id, non_member_sp_application_id, non_member_sp_id

resp = scim_post("Groups", {
    "schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"],
    "displayName": PII_GROUP_NAME,
})
if resp.status_code == 201:
    pii_group_id = resp.json()["id"]
else:
    pii_group_id = scim_get("Groups", params={"filter": f'displayName eq "{PII_GROUP_NAME}"'}).json()["Resources"][0]["id"]

user_lookup = scim_get("Users", params={"filter": f'userName eq "{CALLER_USER}"'})
my_user_id = user_lookup.json()["Resources"][0]["id"]
scim_patch(f"Groups/{pii_group_id}", {
    "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
    "Operations": [{"op": "add", "path": "members", "value": [{"value": my_user_id}]}],
})

sp_resp = scim_post("ServicePrincipals", {
    "schemas": ["urn:ietf:params:scim:schemas:core:2.0:ServicePrincipal"],
    "displayName": SP_DISPLAY_NAME,
    "active": True,
})
if sp_resp.status_code == 201:
    sp_payload = sp_resp.json()
else:
    sp_payload = scim_get("ServicePrincipals", params={"filter": f'displayName eq "{SP_DISPLAY_NAME}"'}).json()["Resources"][0]
non_member_sp_id = sp_payload["id"]
non_member_sp_application_id = sp_payload.get("applicationId")
```

Then generate the OAuth secret in the account console at `/account/serviceprincipals/<application_id>/credentials`. The SCIM API does not expose secret CREATE.

</details>

**Wallet check.** SCIM calls are free. Cumulative session damage: still under five cents on the warehouse side, zero on the account side.

## Task 2: Create the test table with three PII columns and tag them pii=true

Six columns, fifty rows. Three of them carry sensitive values; three do not. After the INSERT, tag the three PII columns with `pii=true` so the ABAC policy in Task 3 has a predicate to fire on.

The non-PII columns (`customer_id`, `signup_date`, `lifetime_value_usd`) are deliberately untagged. The validator confirms BOTH presence on the PII columns AND absence on the non-PII columns; if you accidentally tag `customer_id`, the lab fails because the masking would fire on the wrong column.

In [ ]:
# NBVAL_SKIP
# Task 2: Create schema + table with 50 rows + tag three PII columns.

# YOUR CODE: CREATE SCHEMA IF NOT EXISTS SCHEMA_FQN.
# YOUR CODE: ALTER SCHEMA SCHEMA_FQN SET TAGS for the lab tag.

# YOUR CODE: CREATE TABLE TABLE_FQN with columns:
#   customer_id INT, email STRING, phone STRING, dob DATE,
#   signup_date DATE, lifetime_value_usd DOUBLE
# USING DELTA. Tag the table with the lab tag via ALTER TABLE.

# YOUR CODE: INSERT 50 rows of synthetic data via range(1, 51). Each row:
#   - customer_id = id
#   - email = concat('user_', id, '@example.com')
#   - phone = concat('555-01-', lpad(cast(id as string), 4, '0'))
#   - dob = date_add('1980-01-01', cast(id as int))
#   - signup_date = date_add('2024-01-01', cast(id as int))
#   - lifetime_value_usd = id * 12.5
# Use INSERT INTO ... SELECT ... FROM range(1, 51).

# YOUR CODE: Tag email, phone, dob with pii=true.
# for col in PII_COLUMNS:
#     run_sql(f"ALTER TABLE {TABLE_FQN} ALTER COLUMN {col} SET TAGS ('pii' = 'true')")

print(f"Test table: {TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Table exists with 6 columns and 50 rows. Three PII columns
# carry pii=true; three non-PII columns carry no tag.


def checkpoint_2(session):
    try:
        cat, par, tbl = TABLE_FQN.split(".")
        sql = (
            "SELECT 1 FROM system.information_schema.tables "
            f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
            f"AND table_name = '{tbl}'"
        )
        if not run_sql(sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Table {TABLE_FQN} not found.",
            )

        count = run_sql(f"SELECT COUNT(*) FROM {TABLE_FQN}")["rows"][0][0]
        if int(count) != 50:
            return CheckpointResult(
                status="fail",
                error_reason=f"{TABLE_FQN} has {count} rows; expected exactly 50.",
            )

        col_sql = (
            "SELECT column_name FROM system.information_schema.columns "
            f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
            f"AND table_name = '{tbl}'"
        )
        cols = {r[0] for r in run_sql(col_sql)["rows"]}
        for required in PII_COLUMNS + NON_PII_COLUMNS:
            if required not in cols:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Column {required!r} not found on {TABLE_FQN}. "
                        f"All six columns are required."
                    ),
                )

        # Three PII columns must carry pii=true.
        for pii_col in PII_COLUMNS:
            tag_sql = (
                "SELECT tag_value FROM system.information_schema.column_tags "
                f"WHERE catalog_name = '{cat}' AND schema_name = '{par}' "
                f"AND table_name = '{tbl}' AND column_name = '{pii_col}' "
                f"AND tag_name = 'pii'"
            )
            if not any(r[0] == "true" for r in run_sql(tag_sql)["rows"]):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Column {pii_col!r} is missing tag pii=true. Run "
                        f"ALTER TABLE ... ALTER COLUMN ... SET TAGS."
                    ),
                )

        # Three non-PII columns must NOT carry the pii tag.
        for non_pii_col in NON_PII_COLUMNS:
            tag_sql = (
                "SELECT tag_value FROM system.information_schema.column_tags "
                f"WHERE catalog_name = '{cat}' AND schema_name = '{par}' "
                f"AND table_name = '{tbl}' AND column_name = '{non_pii_col}' "
                f"AND tag_name = 'pii'"
            )
            if run_sql(tag_sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Column {non_pii_col!r} carries tag pii=...; the non-PII "
                        f"columns must be untagged so the masking policy does not "
                        f"fire on them. UNSET the tag."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is wrong: missing table, wrong row count, missing columns, missing PII tag, or accidental tag on a non-PII column. Each is a separate SQL.

</details>

<details><summary>Hint 2 (direction)</summary>

CREATE TABLE with the six columns USING DELTA, INSERT 50 rows from `range(1, 51)`, then loop `PII_COLUMNS` and ALTER each. Tag the table itself with the lab tag too so the orphan scan finds it. Do NOT tag any of the non-PII columns.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE OR REPLACE TABLE {TABLE_FQN} ("
    f"  customer_id INT, email STRING, phone STRING, dob DATE, "
    f"  signup_date DATE, lifetime_value_usd DOUBLE"
    f") USING DELTA"
)
run_sql(f"ALTER TABLE {TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"INSERT INTO {TABLE_FQN} "
    f"SELECT CAST(id AS INT), "
    f"       concat('user_', id, '@example.com'), "
    f"       concat('555-01-', lpad(cast(id as string), 4, '0')), "
    f"       date_add(DATE'1980-01-01', cast(id as int)), "
    f"       date_add(DATE'2024-01-01', cast(id as int)), "
    f"       id * 12.5 "
    f"FROM range(1, 51)"
)

for col in PII_COLUMNS:
    run_sql(f"ALTER TABLE {TABLE_FQN} ALTER COLUMN {col} SET TAGS ('pii' = 'true')")
```

</details>

**Wallet check.** Still about $0.05 cumulative. CREATE TABLE, INSERT 50 rows, three ALTER TABLE ALTER COLUMN; the warehouse handled all of it in seconds.

## Task 3: Create the metastore-scoped ABAC column-mask policy

The headline pattern: one policy that fires when a column carries the `pii=true` tag AND the querying principal is NOT in `pii_approved`. The masked return value is the literal string `***REDACTED***` for the PII columns; the non-PII columns are unaffected because the policy's tag predicate does not match them.

The exact SQL syntax for the metastore-level ABAC policy is what the May 2026 documentation calls the canonical form:

```sql
CREATE OR REPLACE POLICY labrun-pii-mask-policy
ON CATALOG <catalog>
COMMENT 'mask pii=true columns for non-pii_approved'
FOR TABLES
WHEN (HAS_TAG('pii', 'true'))
AS COLUMN MASK
RETURN CASE
  WHEN is_account_group_member('pii_approved') THEN VALUE
  ELSE '***REDACTED***'
END;
```

The validator does not parse the policy DSL strictly because the syntax is still evolving across runtimes; it asserts the policy NAME exists, the policy TYPE includes COLUMN_MASK, and the policy body string contains the substrings `pii` and `pii_approved`. If your runtime uses a slightly different keyword (e.g., `CREATE COLUMN MASK POLICY` instead of `CREATE POLICY ... AS COLUMN MASK`), use the form your workspace accepts and ensure the listing query in `system.information_schema.policies` returns a row.

In [ ]:
# NBVAL_SKIP
# Task 3: Create the ABAC column-mask policy.

# YOUR CODE: CREATE OR REPLACE POLICY ABAC_POLICY_NAME on CATALOG.
# Use the syntax for your Premium workspace's runtime. The validator only
# checks that the policy NAME exists, TYPE includes COLUMN_MASK, and the
# body references both 'pii' and 'pii_approved' as substrings.
# Example (approximate; adjust keywords if your runtime requires it):
#
#   policy_sql = f"""
#   CREATE OR REPLACE POLICY {ABAC_POLICY_NAME}
#   ON CATALOG {CATALOG}
#   COMMENT 'mask pii=true columns for non-pii_approved'
#   FOR TABLES
#   WHEN (HAS_TAG('pii', 'true'))
#   AS COLUMN MASK
#   RETURN CASE
#     WHEN is_account_group_member('{PII_GROUP_NAME}') THEN VALUE
#     ELSE '***REDACTED***'
#   END
#   """
#   run_sql(policy_sql)

print(f"ABAC policy: {ABAC_POLICY_NAME}")
print("Wait a few seconds for the policy to take effect before running Task 4.")
time.sleep(10)

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Policy named ABAC_POLICY_NAME exists in
# system.information_schema.policies; policy_type contains COLUMN_MASK; the
# policy body string references both 'pii' and 'pii_approved'.


def checkpoint_3(session):
    try:
        # Some runtimes expose policies under different views. Try the
        # canonical view first; fall back to a SHOW POLICIES variant if
        # information_schema.policies is unavailable.
        res = run_sql(
            "SELECT policy_name, policy_type, policy_body "
            "FROM system.information_schema.policies "
            f"WHERE policy_name = '{ABAC_POLICY_NAME}'",
            ignore_errors=True,
        )
        if res.get("error"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "system.information_schema.policies is not readable: "
                    f"{res['error']}. Confirm you are signed in as a metastore "
                    f"admin on the trial."
                ),
            )
        rows = res.get("rows") or []
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Policy {ABAC_POLICY_NAME!r} not found in policies view. "
                    f"Run CREATE OR REPLACE POLICY first."
                ),
            )
        name, ptype, pbody = rows[0]
        if "COLUMN_MASK" not in (ptype or "").upper():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Policy type is {ptype!r}; expected to contain COLUMN_MASK. "
                    f"Confirm the policy was created AS COLUMN MASK."
                ),
            )
        body_lower = (pbody or "").lower()
        for required in ("pii", "pii_approved"):
            if required not in body_lower:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Policy body does not reference {required!r}. The policy "
                        f"must predicate on the pii tag AND the pii_approved group."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: the policy row, the COLUMN_MASK type, or the body substrings. Each is a single SQL fragment.

</details>

<details><summary>Hint 2 (direction)</summary>

The policy must (a) name itself exactly `labrun-pii-mask-policy`, (b) be a COLUMN_MASK type, (c) reference `pii` (the tag key) and `pii_approved` (the group) in its body. The exact keyword sequence depends on your runtime; the most common forms are `CREATE POLICY ... AS COLUMN MASK` and `CREATE COLUMN MASK POLICY ...`. Try one; if your runtime errors, try the other.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
policy_sql = f'''
CREATE OR REPLACE POLICY {ABAC_POLICY_NAME}
ON CATALOG {CATALOG}
COMMENT 'mask pii=true columns for non-pii_approved'
FOR TABLES
WHEN (HAS_TAG('pii', 'true'))
AS COLUMN MASK
RETURN CASE
  WHEN is_account_group_member('{PII_GROUP_NAME}') THEN VALUE
  ELSE '***REDACTED***'
END
'''
run_sql(policy_sql)
```

If your runtime rejects the `AS COLUMN MASK` form, try:

```sql
CREATE OR REPLACE COLUMN MASK POLICY labrun-pii-mask-policy
ON CATALOG workspace
WHEN HAS_TAG('pii', 'true')
RETURN CASE WHEN is_account_group_member('pii_approved') THEN VALUE ELSE '***REDACTED***' END
```

</details>

**Wallet check.** Still under $0.10 cumulative. The policy itself is metadata; the SQL warehouse handled the CREATE in well under a second.

## Task 4: Query as a member (this notebook) and as a non-member (the SP), prove masking fires only for the non-member

Two SELECTs against the same table from two principal contexts.

**Member query.** Run the SELECT in this notebook (your workspace PAT authenticates you as a `pii_approved` member). All six columns return clear-text values. Capture the row in `member_result`.

**Non-member query.** Open a `databricks-sql-connector` connection authenticated as the SP (using the OAuth M2M client_id + client_secret you generated in Task 1). Run the same SELECT. The three PII columns return the masked value (`***REDACTED***` per the policy); the three non-PII columns return clear-text values. Capture the row in `non_member_result`.

You will paste two getpass prompts: the SP's client_id (its `applicationId`) and client_secret. The notebook does NOT echo the secret.

The validator compares the two captures column by column:

- Non-PII columns: same value across both contexts.
- PII columns: clear text in member context, masked in non-member context.

In [ ]:
# NBVAL_SKIP
# Task 4: Run the same SELECT as the student (member) and as the SP
# (non-member); capture both rows; INSERT the captured pair so the
# validator can read them.

global member_result, non_member_result

select_sql = (
    f"SELECT customer_id, email, phone, dob, signup_date, lifetime_value_usd "
    f"FROM {TABLE_FQN} ORDER BY customer_id LIMIT 1"
)

# Member query (this notebook is authenticated as the student, who IS in
# pii_approved). Capture columns -> values into member_result.
member_run = run_sql(select_sql)
if not member_run.get("rows"):
    raise RuntimeError("Member query returned no rows. Did you INSERT 50 rows in Task 2?")
member_result = dict(zip(member_run["columns"], [str(v) for v in member_run["rows"][0]]))
print("Member (you) row:")
for k, v in member_result.items():
    print(f"  {k:24s} = {v}")
print()

# Non-member query: authenticate as the SP using OAuth M2M via
# databricks-sql-connector.
sp_client_id = getpass.getpass("Service principal OAuth client_id (= applicationId): ").strip()
sp_client_secret = getpass.getpass("Service principal OAuth client_secret: ").strip()
if not sp_client_id or not sp_client_secret:
    raise RuntimeError("client_id and client_secret are both required.")

# YOUR CODE: Open a databricks.sql.connect() connection using
# OAuth M2M credentials, run select_sql against the same warehouse, and
# capture columns -> values into non_member_result.
# Example:
#   from databricks import sql as dbsql
#   from databricks.sdk.core import Config, oauth_service_principal
#   sp_config = Config(
#       host=databricks_host, client_id=sp_client_id, client_secret=sp_client_secret,
#   )
#   def credentials_provider():
#       return oauth_service_principal(sp_config)
#   with dbsql.connect(
#       server_hostname=databricks_host.replace("https://", ""),
#       http_path=WAREHOUSE_HTTP_PATH,
#       credentials_provider=credentials_provider,
#   ) as conn:
#       with conn.cursor() as cursor:
#           cursor.execute(select_sql)
#           cols = [d[0] for d in cursor.description]
#           row = cursor.fetchone()
#           non_member_result = dict(zip(cols, [str(v) for v in row]))

if non_member_result is None:
    print("non_member_result is None. Open the SP-authenticated connection above.")
    raise SystemExit(1)

print("Non-member (SP) row:")
for k, v in non_member_result.items():
    print(f"  {k:24s} = {v}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: member_result and non_member_result both exist; non-PII
# columns match across both contexts; PII columns differ (clear in member,
# masked in non-member).


def checkpoint_4(session):
    try:
        if not member_result:
            return CheckpointResult(
                status="fail",
                error_reason="member_result is empty. Run the member SELECT first.",
            )
        if not non_member_result:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "non_member_result is empty. Open a SP-authenticated connection "
                    "and run the same SELECT."
                ),
            )

        for col in NON_PII_COLUMNS:
            if member_result.get(col) != non_member_result.get(col):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Non-PII column {col!r} differs across contexts: "
                        f"member={member_result.get(col)!r}, "
                        f"non_member={non_member_result.get(col)!r}. The ABAC policy "
                        f"should not mask non-PII columns."
                    ),
                )

        for col in PII_COLUMNS:
            member_val = member_result.get(col, "")
            non_member_val = non_member_result.get(col, "")
            if member_val == non_member_val:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"PII column {col!r} returned identical values in both "
                        f"contexts ({member_val!r}). The ABAC policy did not fire "
                        f"for the non-member. Confirm: (a) the SP is NOT in "
                        f"pii_approved, (b) the column tag pii=true is present, "
                        f"(c) the policy CREATE succeeded, (d) you waited 10+ "
                        f"seconds after policy CREATE for it to take effect."
                    ),
                )
            if "REDACTED" not in non_member_val.upper() and non_member_val != "***":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Non-member value for {col!r} is {non_member_val!r}; "
                        f"expected the masked literal (e.g., '***REDACTED***'). "
                        f"Check the policy's RETURN clause."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: a captured row, a non-PII column drift, a PII column not differing, or a non-member value that is not the redacted literal. Each maps to a separate fix.

</details>

<details><summary>Hint 2 (direction)</summary>

`databricks-sql-connector` connects via `dbsql.connect(server_hostname=..., http_path=..., credentials_provider=...)`. For OAuth M2M, build a `Config(host=..., client_id=..., client_secret=...)` and wrap `oauth_service_principal(config)` in a `credentials_provider()` callable. Pass that into `connect()`. Run `cursor.execute(select_sql)`, `fetchone()`, and zip with `cursor.description`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
from databricks import sql as dbsql
from databricks.sdk.core import Config, oauth_service_principal

sp_config = Config(
    host=databricks_host,
    client_id=sp_client_id,
    client_secret=sp_client_secret,
)

def credentials_provider():
    return oauth_service_principal(sp_config)

server_hostname = databricks_host.replace("https://", "")
with dbsql.connect(
    server_hostname=server_hostname,
    http_path=WAREHOUSE_HTTP_PATH,
    credentials_provider=credentials_provider,
) as conn:
    with conn.cursor() as cursor:
        cursor.execute(select_sql)
        cols = [d[0] for d in cursor.description]
        row = cursor.fetchone()
        non_member_result = dict(zip(cols, [str(v) for v in row]))
```

If the non-member values still come back clear-text, wait 30 seconds and re-run; ABAC policy propagation has a few-second delay.

</details>

**Wallet check.** $0.10 to $0.50 cumulative for the session. Two SQL warehouse SELECTs plus the SP connection setup. The bigger line item is still the trial-workspace tear-down before day 14.

## Cleanup

The cleanup cell drops the ABAC policy first (a stale policy keeps applying silently), unsets the three column tags, drops the table and schema, then deletes the SP and the group via the SCIM API.

What the cleanup cell handles:

- ABAC policy, column tags, table, schema (all via SQL).
- Service principal `labrun-pii-test-principal` (via SCIM DELETE).
- Group `pii_approved` (via SCIM DELETE).

Manual reminders:

- Confirm in the Databricks account console that no `labrun-` SP or group remains; SCIM DELETE is async and can lag a few seconds.
- Tear down the Premium trial workspace before day 14 of the trial.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down everything in CLEANUP_MANIFEST.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")
print()
print("MANUAL CLEANUP REMINDERS:")
print(f"  1. Confirm in the account console that no labrun- SP or group remains;")
print(f"     SCIM DELETE is async and can lag a few seconds.")
print(f"  2. Tear down the Premium trial workspace before day 14 of the trial,")
print(f"     or it converts to paid and starts billing your cloud account.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.10 to $0.50 on your cloud account.** The Premium trial itself stayed at $0; the warehouse spend is the actual line item. Two SP credentials live in your local memory until you close the notebook tab; never paste them into a chat or a public commit.

## Reflection

These are not graded. They are for you.

1. Walk through what Unity Catalog does when a non-member principal runs `SELECT email FROM customer_records`. Name each step: the column resolution, the tag lookup, the policy evaluation, the principal-group membership check, the mask application. At which step does the policy decide to return `***REDACTED***` vs the clear value?

2. Your team needs to extend this pattern to 40 tables. With ABAC, what is the per-table effort? With per-table column-mask functions, what would the per-table effort be? Sketch the differential maintenance cost over a year, including the case where a new table is added with PII columns the team forgot to tag.

## Exam-Style Practice

**Question 1 (Domain 7, ABAC vs column-mask functions):**

A compliance team wants to mask PII columns (email, phone, dob) across 40 tables in a Unity Catalog metastore. They want to define the masking logic in ONE place and have it apply to every table that has a column matching the criteria, both today and for future tables that get added. Which approach fits?

A. Per-table column-mask functions; write 40 functions and attach each to its table's column.
B. A metastore-level Unity Catalog ABAC policy that fires when a column is tagged `pii=true` and the principal is not in the `pii_approved` group; tag the relevant columns in each table.
C. A bucket-policy-style deny on the S3 paths of the 40 tables.
D. A separate SQL warehouse per principal group; route queries to the right warehouse to enforce masking.

<details><summary>Show answer</summary>

**Correct: B.**

- A works but requires 40 per-table functions plus 40 per-table attachments; adding a new table requires writing another function. This is the maintenance burden the scenario explicitly rules out.
- B is correct: ABAC policies are metastore-scoped and tag-driven. One policy plus per-column tagging delivers the desired behavior across every table that carries the tag pattern. New tables inherit the policy automatically by tagging their PII columns.
- C is wrong: S3 bucket policies cannot mask column values; they can only block or allow object reads. The masking must happen at the query layer, not the storage layer.
- D is wrong: routing queries by principal to different warehouses does not enforce masking; the queries see the same data either way unless a row-level or column-level policy is in place.

</details>

**Question 2 (Domain 7, removing a column tag):**

A platform engineer removes the `pii=true` tag from the `email` column on a table that is governed by a metastore ABAC column-mask policy keyed on `pii=true`. The next query against the table from a non-member principal sees the clear-text email. The engineer expected the policy to keep masking. What happened?

A. ABAC policies cache the tag values for 24 hours; the policy will start masking again after the cache expires.
B. ABAC policies fire based on the current tag state at query time. Removing the tag removes the policy's trigger condition for that column; the policy is still defined but no longer applies to the column.
C. The policy was deleted automatically when the tag was removed; the engineer needs to re-create it.
D. The non-member principal was added to the `pii_approved` group by a different admin; the masking is correctly disabled.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: ABAC policies evaluate at query time against the current metastore state. There is no 24-hour cache.
- B is correct: the policy's predicate is "column has tag pii=true." Removing the tag removes the predicate match; the policy still exists in the metastore but no longer applies to the now-untagged column. To restore masking the engineer must re-set the tag.
- C is wrong: removing a column tag does NOT delete the policy. The policy persists as a metastore object until explicitly DROPped.
- D is wrong: nothing in the scenario suggests group membership changed; the visible behavior is fully explained by the tag removal.

</details>